# Covariables adicionales (revisión del 22-oct-2026: tasa de participación exacta)

Responde los comentarios 15 y 23 del profesor guía: matrícula, tasa de participación, IVE de JUNAEB, pobreza comunal, densidad de población y distancia a la capital regional.

**Estado.** La distancia a la capital regional se calcula con los datos del propio panel y corre de inmediato. Las demás se leen desde `Dataset/Externos/`: IVE e IVM de JUNAEB, resúmenes de matrícula por establecimiento y por curso del MINEDUC y pobreza comunal del Ministerio de Desarrollo Social. Cada sección valida su fuente antes de incorporarla.

In [1]:
import sys, warnings
from pathlib import Path
for m in list(sys.modules):
    if m == "simce_panel" or m.startswith("simce_panel."):
        del sys.modules[m]
candidatos = [Path.cwd() / "simce_panel" / "src", Path.cwd().parent / "src", Path.cwd() / "src"]
sys.path.insert(0, str(next(p for p in candidatos if (p / "simce_panel" / "__init__.py").exists())))
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import simce_panel as sp
sp.configurar_logging()
cfg = sp.Config.desde_base(".")

In [2]:
datos = sp.io_datos.leer_csv(cfg.ruta_salida("dataset_analitico_mate.csv"))
ancho = sp.io_datos.leer_csv(cfg.ruta_salida("simce_rbd_ancho_reducido_2015_2025.csv"))
EXT = Path("Dataset/Externos"); EXT.mkdir(parents=True, exist_ok=True)
C = sp.covariables
print("Dataset analítico:", datos.shape)

Dataset analítico: (3144, 40)


## 1. Distancia a la capital regional (sin datos externos)

La posición de la capital se aproxima por el centroide de los establecimientos de su comuna. Incluye un control de coherencia: fuera de Valparaíso y Magallanes, ningún establecimiento debería estar a más de 800 km de su capital; un valor mayor delata una coordenada errónea.

In [3]:
datos["dist_capital_regional_km"] = C.distancia_capital_regional(datos)
sospechosos = datos["dist_capital_regional_km"].attrs.get("sospechosos")
print(datos["dist_capital_regional_km"].describe().round(1).to_string())
if sospechosos is not None and sospechosos.any():
    print("\nCoordenadas a revisar:")
    print(datos.loc[sospechosos, ["rbd", "nom_rbd", "region_dir", "dist_capital_regional_km"]])

count    3144.0
mean       35.3
std       120.2
min         0.1
25%         5.2
50%        14.0
75%        46.3
max      3676.2


**Acción.** Los establecimientos marcados deben corregirse en el saneamiento de coordenadas (notebook 01) o excluirse del análisis espacial, y documentarse en la tabla de calidad de datos.

## 2. Tasa de participación (comentario 15, alternativa ii)

**Fuente.** Resumen de Matrícula por Curso del MINEDUC (`datosabiertos.mineduc.cl`), un archivo por año con corte al 30 de abril, en `Dataset/Externos/Resumen-de-matricula-por-curso-<año>/`. Se cuenta la matrícula de 2.º medio de niños y jóvenes (`COD_ENSE` 310, 410, 510, 610, 710, 810 y 910 con `COD_GRADO` = 2) de los establecimientos funcionando, sin aulas hospitalarias. Los códigos están en `ER_Matricula_por_Curso.pdf`; la lectura está en `simce_panel/matricula_curso.py`.

**Tasa.** Evaluados en 2.º medio ÷ matrícula de 2.º medio. Reemplaza a la versión anterior, que aproximaba la matrícula de 2.º medio con un cuarto de la de media del resumen por establecimiento; esa aproximación dejaba cerca del 10 % de los registros con tasa sobre 1,05.

**Por qué su nivel no llega a 1.** La matrícula se mide en abril y el SIMCE se rinde a fin de año: la tasa deja fuera a quienes se retiraron entre ambas fechas y a quienes faltaron el día de la prueba. Como esa diferencia se repite cada año, el contraste usa el **cambio** de cada establecimiento.


In [4]:
import re
archivos = sorted(EXT.glob("Resumen_Matricula_EE_Oficial_*.csv"))
partes = []
for f in archivos:
    anio = int(re.search(r"(20\d{2})", f.name).group(1))
    if anio in (2015, 2016, 2017, 2018, 2022, 2023, 2024, 2025):
        partes.append(C.procesar_resumen_matricula(str(f), anio))
mat = pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()
print("Años leídos:", sorted(mat["anio"].unique()) if len(mat) else "ninguno")

15:22:57 | INFO    | simce_panel.covariables | Resumen de matrícula 2015: 2939 establecimientos con media de jóvenes (separador ';').


15:22:57 | INFO    | simce_panel.covariables | Resumen de matrícula 2016: 2948 establecimientos con media de jóvenes (separador ';').


15:22:57 | INFO    | simce_panel.covariables | Resumen de matrícula 2017: 2969 establecimientos con media de jóvenes (separador ';').


15:22:57 | INFO    | simce_panel.covariables | Resumen de matrícula 2018: 2993 establecimientos con media de jóvenes (separador ';').


15:22:57 | INFO    | simce_panel.covariables | Resumen de matrícula 2022: 3038 establecimientos con media de jóvenes (separador ';').


15:22:57 | INFO    | simce_panel.covariables | Resumen de matrícula 2023: 3058 establecimientos con media de jóvenes (separador ';').


15:22:57 | INFO    | simce_panel.covariables | Resumen de matrícula 2024: 3070 establecimientos con media de jóvenes (separador ';').


15:22:57 | INFO    | simce_panel.covariables | Resumen de matrícula 2025: 3091 establecimientos con media de jóvenes (separador ',').


Años leídos: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


In [5]:
# Tasa exacta: evaluados / matrícula de 2.º medio de jóvenes (Resumen de Matrícula por Curso, corte 30 de abril)
from simce_panel.matricula_curso import matricula_2m, participacion_exacta

mat2m = matricula_2m(EXT)            # busca los ocho CSV dentro de Externos/Resumen-de-matricula-por-curso-<año>/
part_todo = participacion_exacta(ancho, mat2m).merge(ancho[["rbd", "cod_grupo_cat"]], on="rbd")

# Control frente a la aproximación anterior (¼ de la matrícula de media)
aprox = part_todo.merge(mat2m[["rbd", "anio", "mat_media"]], on=["rbd", "anio"])
print(f"Registros con tasa > 1,05: exacta {(part_todo.tasa_participacion > 1.05).mean():.1%}"
      f" · aproximada {(aprox.evaluados / (aprox.mat_media / 4) > 1.05).mean():.1%}")

part = part_todo[part_todo["tasa_participacion"].between(0.2, 1.5)]   # descarta errores de registro
mediana = part.groupby("anio")["tasa_participacion"].median().round(3)
print("\nMediana de la tasa exacta por año:"); print(mediana.to_string())


15:22:58 | INFO    | simce_panel.matricula_curso | Matrícula por curso 2015: 2936 establecimientos con media de jóvenes; 2.º medio = 231458


15:22:59 | INFO    | simce_panel.matricula_curso | Matrícula por curso 2016: 2947 establecimientos con media de jóvenes; 2.º medio = 234195


15:23:00 | INFO    | simce_panel.matricula_curso | Matrícula por curso 2017: 2968 establecimientos con media de jóvenes; 2.º medio = 230579


15:23:01 | INFO    | simce_panel.matricula_curso | Matrícula por curso 2018: 2969 establecimientos con media de jóvenes; 2.º medio = 234658


15:23:02 | INFO    | simce_panel.matricula_curso | Matrícula por curso 2022: 2996 establecimientos con media de jóvenes; 2.º medio = 241265


15:23:03 | INFO    | simce_panel.matricula_curso | Matrícula por curso 2023: 3008 establecimientos con media de jóvenes; 2.º medio = 248732


15:23:04 | INFO    | simce_panel.matricula_curso | Matrícula por curso 2024: 3017 establecimientos con media de jóvenes; 2.º medio = 254785


15:23:05 | INFO    | simce_panel.matricula_curso | Matrícula por curso 2025: 3034 establecimientos con media de jóvenes; 2.º medio = 259517


15:23:05 | INFO    | simce_panel.matricula_curso | Tasa exacta: 23533 registros; 184 (0.8 %) sobre 1,05


Registros con tasa > 1,05: exacta 0.8% · aproximada 9.8%

Mediana de la tasa exacta por año:
anio
2015    0.877
2016    0.867
2017    0.882
2018    0.867
2022    0.803
2023    0.826
2024    0.820
2025    0.800


In [6]:
tabla = part.groupby(["cod_grupo_cat", "anio"])["tasa_participacion"].median().unstack().round(3)
display(tabla.reindex(["Bajo", "Medio bajo", "Medio", "Medio alto", "Alto"]))
pre = part[part.anio.between(2015, 2018)].groupby("rbd")["tasa_participacion"].mean()
p22 = part[part.anio == 2022].set_index("rbd")["tasa_participacion"]
cambio = (p22 - pre).rename("cambio_participacion_2022").dropna()
datos = datos.drop(columns=["cambio_participacion_2022"], errors="ignore").merge(
    cambio, left_on="rbd", right_index=True, how="left")
print("Cambio mediano de la participación en 2022, por GSE:")
print(datos.groupby("cod_grupo_cat")["cambio_participacion_2022"].median()
      .reindex(["Bajo", "Medio bajo", "Medio", "Medio alto", "Alto"]).round(3).to_string())

anio,2015,2016,2017,2018,2022,2023,2024,2025
cod_grupo_cat,,,,,,,,
Bajo,0.825,0.811,0.828,0.807,0.750,0.771,0.758,0.727
Medio bajo,0.860,0.845,0.864,0.841,0.783,0.806,0.795,0.772
Medio,0.899,0.884,0.900,0.892,0.825,0.844,0.839,0.816
Medio alto,0.918,0.909,0.924,0.911,0.838,0.867,0.867,0.849
Alto,0.904,0.897,0.918,0.903,0.852,0.864,0.867,0.847


Cambio mediano de la participación en 2022, por GSE:
cod_grupo_cat
Bajo         -0.070
Medio bajo   -0.064
Medio        -0.064
Medio alto   -0.066
Alto         -0.045


**Contraste formal de la alternativa (ii).** Si la participación de 2022 cayó más en los GSE bajos, faltaron en esos colegios los estudiantes de menor desempeño, su promedio quedó inflado y su caída, subestimada: eso explicaría el gradiente no monótono de H3b sin que el aprendizaje del grupo Bajo se hubiera protegido.

El contraste usa el **cambio dentro de cada establecimiento** entre 2018 y 2022. La alternativa se sostiene si los grupos Medio bajo a Alto muestran diferencias **positivas** respecto del Bajo (es decir, si el Bajo perdió más participación).


In [7]:
contraste = C.contraste_participacion(part, ancho[["rbd", "cod_grupo_cat"]],
                                     anio_base=2018, anio_post=2022)
display(contraste["por_gse"])
print("Diferencia respecto del grupo Bajo [estimación, IC 95 %]:")
for g, v in contraste["diferencia_vs_bajo_[est,ic_inf,ic_sup]"].items():
    print(f"  {g:10s} {v[0]:+.3f} [{v[1]:+.3f}; {v[2]:+.3f}]")

,count,median,mean
gse,,,
Bajo,577,-0.062,-0.054
Medio bajo,801,-0.057,-0.058
Medio,680,-0.064,-0.062
Medio alto,356,-0.066,-0.069
Alto,433,-0.041,-0.049


Diferencia respecto del grupo Bajo [estimación, IC 95 %]:
  Medio bajo -0.004 [-0.019; +0.012]
  Medio      -0.008 [-0.023; +0.007]
  Medio alto -0.014 [-0.034; +0.005]
  Alto       +0.006 [-0.012; +0.024]


**Participación entre 2022 y 2025 (matiz para H3).** Si entre 2022 y 2025 la participación del GSE Bajo bajó más que la del Alto, y quienes dejaron de rendir eran estudiantes de menor desempeño, la recuperación del GSE Bajo estaría algo sobrestimada. La celda siguiente mide esa diferencia y acota el sesgo con la ecuación E.20 del informe: sesgo ≈ (1/3)·(f/π)·g, con g entre media y una desviación estándar (25 a 50 puntos).


In [8]:
c25 = C.contraste_participacion(part, ancho[["rbd", "cod_grupo_cat"]], anio_base=2022, anio_post=2025)
display(c25["por_gse"])
print("Diferencia respecto del grupo Bajo, 2022 → 2025 [estimación, IC 95 %]:")
for g, v in c25["diferencia_vs_bajo_[est,ic_inf,ic_sup]"].items():
    print(f"  {g:10s} {v[0]:+.3f} [{v[1]:+.3f}; {v[2]:+.3f}]")
f = c25["por_gse"].loc["Alto", "mean"] - c25["por_gse"].loc["Bajo", "mean"]
pi_bajo = part.loc[(part.anio == 2025) & (part.cod_grupo_cat == "Bajo"), "tasa_participacion"].mean()
for g in (25, 50):
    print(f"Sesgo máximo en la recuperación del GSE Bajo con g = {g}: {f / pi_bajo * g / 3:.2f} puntos por año")


,count,median,mean
gse,,,
Bajo,587,-0.015,-0.019
Medio bajo,833,-0.011,-0.006
Medio,714,-0.007,-0.009
Medio alto,365,0.004,0.008
Alto,445,-0.001,-0.001


Diferencia respecto del grupo Bajo, 2022 → 2025 [estimación, IC 95 %]:
  Medio bajo +0.012 [-0.003; +0.028]
  Medio      +0.010 [-0.005; +0.025]
  Medio alto +0.027 [+0.010; +0.045]
  Alto       +0.017 [-0.001; +0.036]
Sesgo máximo en la recuperación del GSE Bajo con g = 25: 0.21 puntos por año
Sesgo máximo en la recuperación del GSE Bajo con g = 50: 0.42 puntos por año


In [9]:
# Matrícula total (MINEDUC) como covariable de tamaño: a diferencia de la del IVE,
# incluye a los particulares pagados.
mat_tot = (mat[mat.anio.between(2015, 2018)].groupby("rbd")["matricula_total_mineduc"]
           .mean().rename("matricula_total_pre"))
datos = datos.drop(columns=["matricula_total_pre"], errors="ignore").merge(
    mat_tot, left_on="rbd", right_index=True, how="left")
print("Cobertura de la matrícula total previa:", f"{datos['matricula_total_pre'].notna().mean():.1%}")

Cobertura de la matrícula total previa: 95.9%


## 3. Vulnerabilidad escolar de JUNAEB (comentario 23)

Se usan dos productos de `https://www.junaeb.cl/indices-de-vulnerabilidad/`, guardados en `Dataset/Externos/`:

| Índice | Qué mide | Uso en la tesis |
|---|---|---|
| **IVE** | % de estudiantes prioritarios por establecimiento (básica y media juntas); trae la matrícula total | Covariable principal (promedio 2015–2018) |
| **IVM, hoja Media** | Índice multidimensional de enseñanza media | Robustez (no existe para 2015) |

Ambos se fijan al período previo, igual que el GSE. El cambio entre PRE y POST se usa para el contraste composicional.

**Ausencia estructural.** Los particulares pagados no participan de JUNAEB y no tienen IVE. Se les asigna 0 con el indicador `ive_no_aplica`; no se imputan como si faltaran al azar.

In [10]:
panel_j = C.panel_junaeb(EXT)
print(panel_j.groupby("anio").agg(establecimientos=("rbd", "size"),
      ive_medio=("ive", "mean"), ivm_medio=("ivm_media", "mean")).round(3).to_string())

15:23:10 | INFO    | simce_panel.covariables | IVE 2015: hoja 'MEDIA', columna 'IVE-SINAE MEDIA 2015', 2534 establecimientos.


15:23:10 | INFO    | simce_panel.covariables | IVE 2016: hoja 'MEDIA', columna 'IVE-SINAE MEDIA 2016', 2561 establecimientos.


15:23:11 | INFO    | simce_panel.covariables | IVE 2017: hoja 'MEDIA', columna 'IVE-SINAE MEDIA 2017', 2555 establecimientos.


15:23:11 | INFO    | simce_panel.covariables | IVE 2018: hoja 'MEDIA', columna 'IVE-SINAE MEDIA 2018', 2577 establecimientos.


15:23:11 | INFO    | simce_panel.covariables | IVE 2022: hoja 'MEDIA', columna 'IVE-SINAE MEDIA 2022', 2573 establecimientos.


...

15:23:14 | INFO    | simce_panel.covariables | IVM media 2022: hoja 'Media', 2575 establecimientos.


15:23:15 | INFO    | simce_panel.covariables | IVM media 2023: hoja 'Media', 2593 establecimientos.


15:23:15 | INFO    | simce_panel.covariables | IVM media 2024: hoja 'Media', 2610 establecimientos.


15:23:16 | INFO    | simce_panel.covariables | IVM media 2025: hoja 'Media', 2618 establecimientos.


      establecimientos  ive_medio  ivm_medio
anio                                        
2015              2534      0.691        NaN
2016              2631      0.694     20.255
2017              2561      0.697     19.314
2018              2579      0.731     20.888
2022              2575      0.809     18.097
2023              2593      0.793     18.404
2024              2610      0.809     19.510
2025              2618      0.811     20.253


### Validación

Tres controles: (1) correlación con el GSE, que debe ser **negativa y fuerte**; (2) estabilidad entre años consecutivos, que debe ser alta; (3) medias por año sin saltos abruptos. Un salto en la media o una caída en la correlación entre dos años sugiere un **cambio metodológico de JUNAEB**: en ese caso no se deben promediar esos años.

In [11]:
validacion = C.validar_junaeb(panel_j, ancho[["rbd", "cod_grupo_cat"]])
for indice, v in validacion.items():
    print(f"=== {indice}: r con GSE = {v['r_con_gse']}")
    print("  media por año:", v["media_por_anio"])
    print("  r entre años consecutivos:", v["r_entre_anios_consecutivos"])

=== ive: r con GSE = -0.873
  media por año: {2015: 0.691, 2016: 0.694, 2017: 0.697, 2018: 0.731, 2022: 0.809, 2023: 0.793, 2024: 0.809, 2025: 0.811}
  r entre años consecutivos: {'2015-2016': 0.979, '2016-2017': 0.953, '2017-2018': 0.947, '2018-2022': 0.923, '2022-2023': 0.968, '2023-2024': 0.944, '2024-2025': 0.966}
=== ivm_media: r con GSE = -0.516
  media por año: {2016: 20.255, 2017: 19.314, 2018: 20.888, 2022: 18.097, 2023: 18.404, 2024: 19.51, 2025: 20.253}
  r entre años consecutivos: {'2016-2017': 0.886, '2017-2018': 0.958, '2018-2022': 0.738, '2022-2023': 0.887, '2023-2024': 0.914, '2024-2025': 0.921}


In [12]:
resumen_j = C.resumir_junaeb(panel_j)
datos = C.ive_para_modelo(datos, resumen_j.drop(columns=[c for c in resumen_j.columns if c.startswith('matricula_total')]))
print("Cobertura del IVE real por dependencia (%; los pagados tienen 0 asignado e indicador):")
sub = datos[datos.ive_no_aplica == 0]
print((sub.groupby("cod_depe1_cat")["ive_pre"].apply(lambda s: s.notna().mean()) * 100).round(1).to_string())
print("\nIVE previo por GSE (subvencionados):")
print(datos[datos.ive_no_aplica == 0].groupby("cod_grupo_cat")["ive_pre"].median().round(3).to_string())

15:23:16 | INFO    | simce_panel.covariables | IVE previo: 480 pagados con 0 e indicador; 129 subvencionados sin dato.


Cobertura del IVE real por dependencia (%; los pagados tienen 0 asignado e indicador):
cod_depe1_cat
Corporación de administración delegada    100.0
Municipal Corporación                      93.9
Municipal DAEM                             93.7
Particular subvencionado                   95.5
Servicio Local de Educación                94.5

IVE previo por GSE (subvencionados):
cod_grupo_cat
Alto          0.221
Bajo          0.891
Medio         0.625
Medio alto    0.415
Medio bajo    0.788


**Contraste composicional.** `ive_cambio` mide si la vulnerabilidad del establecimiento cambió entre períodos. Si las rupturas sostenidas coincidieran con aumentos del IVE, parte de la caída sería un cambio de población y no de aprendizaje.

In [13]:
if "ive_cambio" in datos.columns and "tipo_trayectoria" in datos.columns:
    print(datos[datos.ive_no_aplica == 0].groupby("tipo_trayectoria")["ive_cambio"]
          .describe()[["count", "50%", "mean"]].round(3).to_string())

                    count    50%   mean
tipo_trayectoria                       
atípico 2022         80.0  0.140  0.126
normal             2124.0  0.086  0.089
ruptura sostenida   200.0  0.152  0.146
sin dato             42.0  0.093  0.098


## 4. Pobreza, población y densidad comunal (comentario 23)

**Fuentes**, en `Dataset/Externos/`:
- `sae_ingresos_<año>.xlsx` (2015, 2017, 2020, 2022, 2024): pobreza por ingresos por comuna, estimación para áreas pequeñas basada en la CASEN, del Observatorio Social del Ministerio de Desarrollo Social y Familia. Cada archivo trae además la población proyectada.
- `Superficie por comuna 2016-2020.csv.csv`: superficie comunal (km²), Biblioteca del Congreso Nacional.

**Decisiones.**
- La covariable de pobreza es el **promedio de 2015 y 2017**, alineada con el período previo, igual que el GSE y el IVE.
- No se usa el cambio de pobreza entre períodos: la estimación 2024 usa la nueva medición de pobreza y su nivel no es comparable con los años anteriores.
- La densidad es la población proyectada de 2017 dividida por la superficie.
- Los códigos de la antigua provincia de Ñuble (84xx) se armonizan por nombre de comuna contra el archivo más reciente.
- `Poblacion_2024.xlsx` no se usa: informa la población por entidad y excluye a la población rural dispersa.

In [14]:
pobreza = C.panel_pobreza(EXT)
print("Años leídos:", sorted(pobreza["anio"].unique()))
print(pobreza.groupby("anio")["pobreza_comunal"].agg(["count", "median"]).round(3).to_string())
print(C.validar_pobreza(datos, pobreza[pobreza["anio"] == pobreza["anio"].max()]))

15:23:16 | INFO    | simce_panel.covariables | Población comunal derivada del número de pobres y la tasa.


15:23:16 | INFO    | simce_panel.covariables | Pobreza comunal: 345 comunas; tasa mediana 0.158.


15:23:16 | INFO    | simce_panel.covariables | Población comunal derivada del número de pobres y la tasa.


15:23:16 | INFO    | simce_panel.covariables | Pobreza comunal: 345 comunas; tasa mediana 0.114.


15:23:16 | INFO    | simce_panel.covariables | Pobreza comunal: 345 comunas; tasa mediana 0.127.


15:23:16 | INFO    | simce_panel.covariables | Pobreza comunal: 345 comunas; tasa mediana 0.086.


15:23:16 | INFO    | simce_panel.covariables | Pobreza comunal: 345 comunas; tasa mediana 0.203.


15:23:16 | INFO    | simce_panel.covariables | Pobreza 2015: 21 códigos comunales armonizados por nombre.


Años leídos: [np.int64(2015), np.int64(2017), np.int64(2020), np.int64(2022), np.int64(2024)]
      count  median
anio               
2015    345   0.158
2017    345   0.114
2020    345   0.127
2022    345   0.086
2024    345   0.203
{'cobertura_establecimientos': 0.999, 'comunas_sin_codigo': [8401], 'r_pobreza_vs_gse_vulnerable': 0.546, 'comunas_comparadas': 335}


**Validación.** Cada año debe cubrir unas 345 comunas, y la correlación entre la pobreza comunal y la proporción de colegios de GSE Bajo o Medio bajo debe ser positiva. Si un año aparece con muchas menos comunas, revisa el nombre de sus columnas.

In [15]:
pre = (pobreza[pobreza["anio"].isin([2015, 2017])]
       .groupby("cut").agg(pobreza_comunal_pre=("pobreza_comunal", "mean"),
                           pobreza_solo_sintetica=("pobreza_solo_sintetica", "max")))
pob17 = pobreza[pobreza["anio"] == 2017].set_index("cut")["poblacion_comunal"].rename("poblacion_2017")
datos["cut"] = pd.to_numeric(datos["comuna_dir"], errors="coerce")
datos = (datos.drop(columns=[c for c in ["pobreza_comunal_pre", "pobreza_solo_sintetica",
                                         "poblacion_2017", "pobreza_comunal", "poblacion_comunal"]
                             if c in datos.columns])
              .merge(pre, left_on="cut", right_index=True, how="left")
              .merge(pob17, left_on="cut", right_index=True, how="left"))
print("Cobertura de la pobreza previa:", f"{datos['pobreza_comunal_pre'].notna().mean():.1%}")

Cobertura de la pobreza previa: 99.9%


In [16]:
ruta_sup = next(EXT.glob("Superficie*"), None)
if ruta_sup is not None:
    referencia = pobreza[pobreza["anio"] == pobreza["anio"].max()][["cut", "nombre"]].dropna()
    superficie = C.leer_superficie(str(ruta_sup), referencia)
    print(C.validar_superficie(superficie))
    datos = (datos.drop(columns=["superficie_km2"], errors="ignore")
                  .merge(superficie, on="cut", how="left"))
    datos["densidad_comunal"] = datos["poblacion_2017"] / datos["superficie_km2"]
    datos["log_densidad_comunal"] = np.log(datos["densidad_comunal"])
    print("Cobertura de la densidad:", f"{datos['densidad_comunal'].notna().mean():.1%}")
    print(datos.groupby("cod_rural_rbd_cat")["densidad_comunal"].median().round(0).to_string())
else:
    print("Falta el archivo de superficie en", EXT)

15:23:16 | INFO    | simce_panel.covariables | Superficie: 335 comunas con código.


{'comunas': 335, 'santiago_km2': 22.0, 'santiago_plausible': True, 'max_km2': 49924.0, 'min_km2': 7.0}
Cobertura de la densidad: 99.0%
cod_rural_rbd_cat
Rural      33.0
Urbano    349.0


**Validación de la superficie.** La comuna de Santiago debe medir unos 22 km² (`santiago_plausible = True`). Un valor diez veces mayor o menor delata un problema de separador decimal. La densidad mediana de los establecimientos urbanos debe ser muy superior a la de los rurales.

## 5. Guardado

In [17]:
datos.to_csv(cfg.ruta_salida("dataset_analitico_mate_covariables.csv"), sep=";",
             index=False, encoding="utf-8-sig")
print("Columnas nuevas:", [c for c in ["dist_capital_regional_km", "ive",
      "pobreza_comunal_pre", "log_densidad_comunal", "pobreza_solo_sintetica", "cambio_participacion_2022", "ive_pre", "ive_cambio", "ivm_media_pre", "matricula_total_pre"] if c in datos.columns])

Columnas nuevas: ['dist_capital_regional_km', 'pobreza_comunal_pre', 'log_densidad_comunal', 'pobreza_solo_sintetica', 'cambio_participacion_2022', 'ive_pre', 'ive_cambio', 'ivm_media_pre', 'matricula_total_pre']
